**Setup ambiente**

In [9]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn

import os, json, numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


**Path + LOAD DATASET (HF datasets) + LABEL MAP**

In [17]:
import os, json
import pandas as pd
from datasets import load_dataset, DatasetDict

BASE = "data/raw"
IN_PATH = os.path.join(BASE, "dataset_it.json")
OUT_DIR = os.path.join(BASE, "dataset_002_it")
os.makedirs(OUT_DIR, exist_ok=True)

train_clf_path = os.path.join(OUT_DIR, "train_classification.json")
test_clf_path  = os.path.join(OUT_DIR, "test_classification.json")

# --- build dataset_002 se manca ---
if not (os.path.exists(train_clf_path) and os.path.exists(test_clf_path)):
    raw_hf = load_dataset("json", data_files=IN_PATH)["train"]

    # split 80/20 SENZA sklearn
    split_pairs = raw_hf.train_test_split(test_size=0.20, seed=42, shuffle=True)
    train_pairs = split_pairs["train"].to_pandas()
    test_pairs  = split_pairs["test"].to_pandas()

    # pulizia base
    for df_ in (train_pairs, test_pairs):
        df_["non_inclusiva"] = df_["non_inclusiva"].astype(str).str.strip()
        df_["inclusiva"] = df_["inclusiva"].astype(str).str.strip()
    train_pairs = train_pairs[(train_pairs["non_inclusiva"]!="") & (train_pairs["inclusiva"]!="")].reset_index(drop=True)
    test_pairs  = test_pairs[(test_pairs["non_inclusiva"]!="") & (test_pairs["inclusiva"]!="")].reset_index(drop=True)

    def make_classification(df_):
        rows = []
        for _, r in df_.iterrows():
            rows.append({"text": r["inclusiva"], "label": "inclusiva"})
            rows.append({"text": r["non_inclusiva"], "label": "non_inclusiva"})
        return pd.DataFrame(rows)

    train_df = make_classification(train_pairs).sample(frac=1, random_state=42).reset_index(drop=True)
    test_df  = make_classification(test_pairs).sample(frac=1, random_state=42).reset_index(drop=True)

    def save_jsonl(df_, path):
        with open(path, "w", encoding="utf-8") as f:
            for _, r in df_.iterrows():
                f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

    save_jsonl(train_df, train_clf_path)
    save_jsonl(test_df,  test_clf_path)

    print("Creati:", train_clf_path, "e", test_clf_path)

# --- load HF datasets ---
ds = load_dataset("json", data_files={"train": train_clf_path, "test": test_clf_path})

# validation dal train (10%)
split = ds["train"].train_test_split(test_size=0.1, seed=42, shuffle=True)
ds = DatasetDict({"train": split["train"], "validation": split["test"], "test": ds["test"]})

print(ds)
print("Esempio:", ds["train"][0])

# --- label map ---
labels = sorted(list(set(ds["train"]["label"])))
LABEL2ID = {lab:i for i, lab in enumerate(labels)}
ID2LABEL = {i:lab for lab,i in LABEL2ID.items()}
print("LABEL2ID:", LABEL2ID)

def encode_label(ex):
    ex["label_id"] = LABEL2ID[ex["label"]]
    return ex

ds = ds.map(encode_label)


FileNotFoundError: Unable to find 'C:/Users/rober/Desktop/Semantic/classificatore\data/raw\dataset_it.json'

**Check leakage**

In [ ]:
train_texts = set(ds["train"]["text"])
val_texts   = set(ds["validation"]["text"])
test_texts  = set(ds["test"]["text"])

print("Overlap train∩val :", len(train_texts & val_texts))
print("Overlap train∩test:", len(train_texts & test_texts))zz
print("Overlap val∩test  :", len(val_texts & test_texts))


**TOKENIZE**

In [ ]:
MODEL_NAME = "indigo-ai/BERTino"   # veloce su T4, ottimo baseline
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized = ds.map(tokenize_fn, batched=True, remove_columns=[c for c in ds["train"].column_names if c not in ["label_id"]])
tokenized = tokenized.rename_column("label_id", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


**TRAIN + METRICHE**

In [ ]:
import numpy as np
import torch

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


# ---- MODEL ----
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# ---- METRICS (robusta: gestisce predictions come tuple/list) ----
def compute_metrics(eval_pred):
    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    # a volte Trainer restituisce (logits, ...) invece dei soli logits
    if isinstance(preds, (tuple, list)):
        preds = preds[0]

    preds = np.argmax(preds, axis=-1)

    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}


# ---- TRAINING ARGS ----
args = TrainingArguments(
    output_dir="./clf_out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# ---- COLLATOR (se non l'hai già creato sopra) ----
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ---- TRAINER ----
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,  # warning futuro ok
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


**Evaluate + Save + Pipeline test**

In [ ]:
import os
import json
import numpy as np
import torch
from sklearn.metrics import classification_report
from transformers import pipeline

# --- evaluate sul test ---
test_metrics = trainer.evaluate(tokenized["test"])
print("TEST METRICS:", test_metrics)

# --- report dettagliato ---
pred = trainer.predict(tokenized["test"])

y_true = pred.label_ids

logits = pred.predictions
# FIX: a volte predictions è (logits, ...) -> prendo solo logits
if isinstance(logits, (tuple, list)):
    logits = logits[0]

y_pred = np.argmax(logits, axis=-1)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[ID2LABEL[i] for i in range(len(ID2LABEL))]
    )
)

# --- save (modello + tokenizer + label map) ---
SAVE_DIR = "./model_classifier_it"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(os.path.join(SAVE_DIR, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump(ID2LABEL, f, indent=2, ensure_ascii=False)

print("Salvato in:", SAVE_DIR)

# --- pipeline test (uso reale) ---
clf = pipeline(
    "text-classification",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1
)

samples = [
    "Le ragazze bionde sono poco intelligenti.",
    "L'intelligenza di una persona non è determinata dal suo colore dei capelli."
]

for s in samples:
    print(s, "->", clf(s))


In [ ]:
# (1) crea lo zip della cartella
!zip -r model_classifier_it.zip model_classifier_it
